# 4.3 Execution State and Recovery on τ²-bench: Does a Checkpoint Graph Let a Crashed Agent Resume Correctly?

*Formerly notebook 27.*

> **Outputs in this copy come from the confidence run** (`SMASH_MODE=batch`): 273 crash points × 4 conditions from 138
> successful reference runs, 12,543 calls at the standard tier, $6.58 (2026-11-11). Task 4.3. They are reported in
> `RESULTS-REPORT.md` and the v2 paper.

**Presentation map:** Task 4.3 (execution state and recovery: plan steps, progress and checkpoints as a graph). Paper v2,
`PLAN.md` §5 row 5.3, §5a, §5b. Published evidence: Crab (arXiv 2604.28138) and AgentRewind (arXiv 2608.14380) compare
restart, resume from the message history and checkpoints, but none stores the checkpoint as a graph or uses τ²-bench.

## Protocol (frozen before the confidence run)

**Research question.** After an agent crashes in the middle of a τ²-bench task, does resuming from a checkpoint graph
(completed steps, the pending step, results and references to the database records) finish the task correctly as often as
resuming from the message history, with less repeated work and fewer tokens, and better than restarting?

**Hypothesis.** H1: resuming from the checkpoint graph (d) succeeds at least as often as resuming from the message
history with the same information (c), with fewer tokens per resumed run (the graph keeps each database record once, at
its latest state, instead of every tool output). H2: both beat restarting from scratch (a) on success and repeated steps
(Crab: history-only resume 8–13 % correct vs. 100 % for checkpoints, on file-system state). H3: the plain message
history (b) repeats or misjudges the step that was in flight when the crash hit, most visibly when it was a write.

**Reference runs.** The 278 τ²-bench recordings of Notebook 4.4 (airline 50, retail 114, telecom 114; commit `96a3ef9`;
`gpt-6-luna` as agent and user simulator). As in Crab, only runs that **succeeded** by τ²-bench's state-based evaluation
are used, so that a failure after the crash is the crash's fault.

**Crash injection.** Two crash points per reference run (one if the run has a single tool step), drawn with a seeded RNG
uniformly among the agent's **tool-call steps**. The crash hits after the environment executed the step and before its
result reached the agent: the database keeps the step's effect, the agent loses it. The user simulator keeps its memory;
the pre-crash part of every resumed run is the recorded reference, identical for all conditions.

**Conditions (what the resumed agent sees besides its system prompt):**
- **(a) Restart from scratch**: nothing; the agent greets the customer again ("we were disconnected"), the customer
  simulator continues.
- **(b) Resume from the message history**: the full message history up to the crashed step (dialogue, tool calls and
  results), as a chat log would persist it.
- **(c) Message history + the same checkpoint facts** (same information as (d), §5b rule 4): (b) plus a note naming the
  step that was in flight and whether the database records it names changed since the step started (from the stored
  state references).
- **(d) Checkpoint graph**: a graph written as the run proceeds: dialogue turns (NEXT edges), step nodes (tool, arguments,
  status completed / failed / pending, result), record nodes (one per database record seen, latest state) linked by
  READ / WROTE edges, and the pending step with STATE_REF edges to the records its arguments name (content when the step
  started). The resumed agent gets the graph serialised as text and the same pending-step note as (c), not the raw
  message history.

All four continue with the same model, prompt, tools and limits (100 steps, 10 tool errors, counted from the resume).

**Primary metric.** Successful resumption = the resumed run ends with τ²-bench state-based reward 1 (DB hash vs. gold
actions in airline and retail, environment assertions in telecom, plus τ²-bench's deterministic communicate / action
checks where the task's reward basis has them; the LLM-judged NL assertions of retail are not run). **Primary
comparison: (d) − (c)**, paired over crash points, 95% paired bootstrap (5,000 resamples). **Secondary:** (d) − (b),
(d) − (a), (c) − (b); repeated steps (post-resume agent tool calls identical to one made before the crash; repeated writes
separately); tokens after the resume (agent + user simulator) and the agent's prompt size at the resume; all split by
whether the crashed step was a write.

**n.** 2 crash points × the successful reference runs (expected ≈ 300–400 crash points × 4 conditions).

**Threats.** Successful reference runs only (as Crab), so the tasks are those the model can solve; the crash model is one
kind (after execution, before the result is stored); the user simulator is an LLM and can drift after a restart; the
checkpoint graph's record extraction uses the domains' id fields (written for these three domains); one model.

**Protocol changes after the test run:** none yet.

In [ ]:
# Notebooks live in examples/<task group>/; data/, smash_batch.py and the .venv-* environments stay in examples/.
# Every path below is relative to examples/, so run from there whether the notebook is opened in its own folder or not.
import os, sys
if not os.path.exists("smash_batch.py") and os.path.exists(os.path.join("..", "smash_batch.py")):
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [1]:
import json, os, re, random, subprocess, hashlib, time
from collections import Counter, defaultdict
import numpy as np
import networkx as nx
from smash_batch import LLM, mode_from_env, paired_boot, wilson

MODE = mode_from_env()               # replay (default) | test (small live run) | batch (full run via Batch API)
TEST = MODE == "test"
BASE = "data/benchmarks/tau2"
llm = LLM(f"{BASE}/cache" + ("_test" if TEST else ""))
PYV = ".venv-tau2/bin/python"
REC_PATH = f"{BASE}/nb28_recordings" + ("_test" if TEST else "") + ".json"
for p in (REC_PATH, f"{BASE}/tau2_env_server.py", PYV):
    assert os.path.exists(p), f"{p} is missing: run Notebook 4.4 first (it downloads τ²-bench, builds its venv and records the reference runs)"
R = json.load(open(REC_PATH))
REC = R["convs"]
TAU2_COMMIT = R["commit"]
N_CRASH = 2
print("MODE =", MODE, "| reference recordings:", len(REC), dict(Counter(c["domain"] for c in REC)), "| tau2-bench", TAU2_COMMIT[:7])

MODE = batch | reference recordings: 278 {'airline': 50, 'retail': 114, 'telecom': 114} | tau2-bench 96a3ef9


## The τ²-bench environment server and the conversation engine

The server is Notebook 4.4's (`data/benchmarks/tau2/tau2_env_server.py`, τ²-bench's own environment code in
`examples/.venv-tau2`). The engine is Notebook 4.4's, with one change: the agent's view can start with a **prefix** (what
the agent was given at the resume) followed by the conversation after the resume.

In [2]:
class Tau2Server:
    def __init__(self):
        self.p = subprocess.Popen([PYV, "-I", f"{BASE}/tau2_env_server.py"], stdin=subprocess.PIPE,
                                  stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
    def ask(self, req):
        self.p.stdin.write(json.dumps(req) + "\n"); self.p.stdin.flush()
        r = json.loads(self.p.stdout.readline())
        if "error" in r:
            raise RuntimeError(r["error"] + "\n" + r.get("trace", ""))
        return r

srv = Tau2Server()
META = srv.ask({"op": "meta"})
DOM = META["domains"]
TASK = {f"{t['domain']}|{t['id']}": t for t in META["tasks"]}
WRITE = {t for d in DOM.values() for t in d["write_tools"]}

In [3]:
STOP_TOKENS = ("###STOP###", "###TRANSFER###", "###OUT-OF-SCOPE###")
GREETING = "Hi! How can I help you today?"          # tau2-bench's fixed first agent message
MAX_STEPS, MAX_ERRORS = 100, 10                      # tau2-bench defaults (messages incl. tool results; tool errors)

def oa_calls(tcs):
    return [{"id": t["id"], "type": "function", "function": {"name": t["name"], "arguments": t["raw"]}} for t in tcs]

def agent_view(conv, events=None):
    """What the agent LLM sees: its policy, the dialogue, and its own tool calls and results (not the user's).
    After a resume: the prefix given at the resume, then the conversation since the resume."""
    msgs = [{"role": "system", "content": DOM[conv["domain"]]["agent_system"]}]
    if events is None:
        events = conv.get("agent_prefix", []) + conv["events"][conv.get("resume_at", 0):]
    for e in events:
        if e["role"] == "assistant":
            m = {"role": "assistant", "content": e["content"]}
            if e["tool_calls"]:
                m["tool_calls"] = oa_calls(e["tool_calls"])
            msgs.append(m)
        elif e["role"] == "user" and not e["tool_calls"]:
            msgs.append({"role": "user", "content": e["content"]})
        elif e["role"] == "tool" and e["requestor"] == "assistant":
            msgs.append({"role": "tool", "tool_call_id": e["id"], "content": e["content"]})
        elif e["role"] == "note":                     # notebook 4.3: resume notes for the agent
            msgs.append({"role": "user", "content": e["content"]})
    return msgs

def user_view(conv):
    """What the user simulator sees (roles flipped, as in tau2-bench): agent messages as 'user', its own as
    'assistant', its own tool calls and results; the agent's tool calls are hidden."""
    msgs = [{"role": "system", "content": TASK[conv["key"]]["user_system"]}]
    for e in conv["events"]:
        if e["role"] == "assistant" and not e["tool_calls"] and e["content"]:
            msgs.append({"role": "user", "content": e["content"]})
        elif e["role"] == "user":
            m = {"role": "assistant", "content": e["content"]}
            if e["tool_calls"]:
                m["tool_calls"] = oa_calls(e["tool_calls"])
            msgs.append(m)
        elif e["role"] == "tool" and e["requestor"] == "user":
            msgs.append({"role": "tool", "tool_call_id": e["id"], "content": e["content"]})
    return msgs

def new_conv(key, cid=None):
    d, t = key.split("|", 1)
    return {"id": cid or key, "key": key, "domain": d, "task": t, "turn": "user", "done": False, "term": None,
            "steps": 1, "errors": 0, "llm_calls": 0, "tokens": {"agent_in": 0, "agent_out": 0, "user_in": 0, "user_out": 0},
            "events": [{"role": "assistant", "content": GREETING, "tool_calls": []}], "log": []}

def parse_calls(r):
    out = []
    for t in r.get("tool_calls") or []:
        try:
            args = json.loads(t["arguments"]) if t["arguments"] else {}
            args = args if isinstance(args, dict) else None
        except json.JSONDecodeError:
            args = None
        out.append({"id": t["id"], "name": t["name"], "raw": t["arguments"], "arguments": args})
    return out

def agent_spoke(c):
    return any(e["role"] == "assistant" and not e["tool_calls"] and e["content"] for e in c["events"][1:])

def body_for(c):
    if c["turn"] == "agent":
        b = LLM.body(agent_view(c), max_tokens=1500, tools=DOM[c["domain"]]["tools"])
    else:   # the user's device tools (telecom) only after the agent has answered once (see the protocol changes)
        b = LLM.body(user_view(c), max_tokens=1000, tools=TASK[c["key"]]["user_tools"] if agent_spoke(c) else None)
    b["seed"] = c.get("seed", 7)                    # recording pass (Notebook 4.4); 7 = smash_batch's default
    return b

# Each lock-step round is one Batch job, and the Batch queue took hours per round (2026-11-09/10). STEP_TIER = "standard"
# sends the rounds synchronously (standard tier, 2x price; same model and settings, tier only), as notebooks 1.2b and 2.3.
# 2026-11-10: set before the full run started. Other stages (verifiers, judges) still use the Batch API.
STEP_TIER = "standard"

def simulate(convs, stage, max_steps=MAX_STEPS, step0=0, on_step=None):
    """Run all conversations in lockstep: every round is ONE llm.run over all live conversations (each needs either
    an agent or a user-simulator turn), so in batch mode each round is one Batch job. Tool calls are executed by
    the tau2 server between rounds."""
    n = step0
    while True:
        live = [c for c in convs if not c["done"]]
        if not live:
            break
        n += 1
        mode0 = llm.mode
        if MODE == "batch" and STEP_TIER == "standard":
            llm.mode = "test"                        # synchronous standard tier for the lock-step rounds only
        try:
            res = llm.run(f"{stage}_step{n}", {c["id"]: body_for(c) for c in live})
        finally:
            llm.mode = mode0
        todo = []
        for c in live:
            r, role = res[c["id"]], c["turn"]
            c["llm_calls"] += 1
            if role == "agent" and "first_agent_in" not in c:
                c["first_agent_in"] = r.get("in", 0)           # prompt size of the agent's first turn (Notebook 4.3)
            c["tokens"][f"{role}_in"] += r.get("in", 0); c["tokens"][f"{role}_out"] += r.get("out", 0)
            if "error" in r:
                c["done"], c["term"] = True, f"{role}_llm_error"; continue
            tcs = parse_calls(r)
            c["events"].append({"role": "assistant" if role == "agent" else "user", "content": r["text"] or None,
                                "tool_calls": tcs})
            c["steps"] += 1
            if tcs:
                todo.append(c)
            elif not r["text"]:
                c["done"], c["term"] = True, f"{role}_error"
            elif role == "user" and any(s in r["text"] for s in STOP_TOKENS):
                c["done"], c["term"] = True, "user_stop"
            else:
                c["turn"] = "user" if role == "agent" else "agent"
        if todo:
            items = [{"domain": c["domain"], "task": c["task"], "log": c["log"],
                      "calls": [{"id": t["id"], "name": t["name"], "arguments": t["arguments"],
                                 "requestor": "assistant" if c["turn"] == "agent" else "user"}
                                for t in c["events"][-1]["tool_calls"] if t["arguments"] is not None]} for c in todo]
            out = srv.ask({"op": "exec", "items": items})["results"]
            for c, it, rs in zip(todo, items, out):
                rq = "assistant" if c["turn"] == "agent" else "user"
                rs = iter(rs)
                for t in c["events"][-1]["tool_calls"]:
                    if t["arguments"] is None:
                        o = {"content": "Error: the tool call arguments are not valid JSON.", "error": True}
                    else:
                        o = next(rs)
                        c["log"].append({"id": t["id"], "name": t["name"], "arguments": t["arguments"], "requestor": rq,
                                         "error": o["error"], "event": len(c["events"]) - 1})
                    c["events"].append({"role": "tool", "id": t["id"], "name": t["name"], "content": o["content"],
                                        "requestor": rq, "error": o["error"]})
                    c["errors"] += o["error"]
                c["steps"] += 1
        for c in live:
            if not c["done"] and c["steps"] >= max_steps:
                c["done"], c["term"] = True, "max_steps"
            elif not c["done"] and c["errors"] >= MAX_ERRORS:
                c["done"], c["term"] = True, "too_many_errors"
        if on_step:
            on_step(n, live)
    return n

def evaluate(convs, extra_events=None):
    """tau2-bench's state-based reward on the environment after every executed tool call (c['log']); conversations
    that did not end with a user stop get 0, as in tau2-bench."""
    items = []
    for c in convs:
        msgs = []
        for e in (extra_events or {}).get(c["id"], []) + c["events"]:
            if e["role"] in ("assistant", "user"):
                msgs.append({"role": e["role"], "content": e["content"],
                             "tool_calls": [{"id": t["id"], "name": t["name"], "arguments": t["arguments"]}
                                            for t in e["tool_calls"] if t["arguments"] is not None]})
            elif e["role"] == "tool":
                msgs.append({"role": "tool", "id": e["id"], "content": e["content"], "requestor": e["requestor"]})
        items.append({"domain": c["domain"], "task": c["task"], "log": c["log"], "messages": msgs})
    out = srv.ask({"op": "evaluate", "items": items})["results"]
    for c, r in zip(convs, out):
        c["eval"] = r
        c["reward"] = r["reward"] if c["term"] == "user_stop" else 0.0

## Reference runs and crash points

In [4]:
REF = [c for c in REC if c["reward"] == 1.0]
print(f"successful reference runs: {len(REF)} of {len(REC)}", dict(Counter(c["domain"] for c in REF)))

def tool_steps(c):
    return [i for i, e in enumerate(c["events"]) if e["role"] == "assistant" and e["tool_calls"]
            and any(t["arguments"] is not None for t in e["tool_calls"])]

CRASH = []
for c in REF:
    steps = tool_steps(c)
    if not steps:
        continue
    rng = random.Random(f"nb27|{c['id']}")
    for e in sorted(rng.sample(steps, min(N_CRASH, len(steps)))):
        calls = [t for t in c["events"][e]["tool_calls"] if t["arguments"] is not None]
        CRASH.append({"cid": f"{c['id']}|e{e}", "ref": c, "e": e, "k": steps.index(e) + 1, "n_steps": len(steps),
                      "calls": calls, "write": any(t["name"] in WRITE for t in calls),
                      "pre_log": [l for l in c["log"] if l["event"] < e],
                      "crash_log": [l for l in c["log"] if l["event"] <= e]})
print(f"crash points: {len(CRASH)} ({sum(x['write'] for x in CRASH)} at a write step); "
      f"mean position: step {np.mean([x['k'] for x in CRASH]):.1f} of {np.mean([x['n_steps'] for x in CRASH]):.1f}")

successful reference runs: 138 of 278 {'airline': 26, 'retail': 82, 'telecom': 30}
crash points: 273 (32 at a write step); mean position: step 4.5 of 8.0


## The checkpoint: pending-step check against the stored state references

When a step starts, the checkpoint stores references to the database records its arguments name, with their content.
On resume, the records are read again and compared: changed means the step took effect. The same note goes to (c) and (d).

In [5]:
NAMED = {}
for x in CRASH:
    for t in x["calls"]:
        it = {"domain": x["ref"]["domain"], "task": x["ref"]["task"], "args": t["arguments"]}
        before, after = (srv.ask({"op": "named", "items": [{**it, "log": lg}]})["results"][0]
                         for lg in (x["pre_log"], x["crash_log"]))
        NAMED[(x["cid"], t["id"])] = (before, after)

def pending_note(x):
    parts = []
    for t in x["calls"]:
        before, after = NAMED[(x["cid"], t["id"])]
        head = f"{t['name']}({t['raw']})"
        if t["name"] not in WRITE:
            parts.append(f"{head}: a read-only step; its result was lost, run it again if you need it.")
        elif not before:
            parts.append(f"{head}: a write step; the checkpoint holds no record it names, so its effect is unknown; "
                         "check the database before repeating it.")
        else:
            ch = [n for n in before if before[n] != after.get(n)]
            parts.append(f"{head}: a write step; database check of {', '.join(sorted(before))} against the checkpoint "
                         f"taken when the step started: " + (f"{', '.join(ch)} changed, so the step took effect."
                                                            if ch else "unchanged, so the step did not take effect."))
    return ("Your session crashed during your last step. The step was sent to the system but its result was lost.\n"
            + "\n".join("Pending step: " + p for p in parts) +
            "\nContinue the conversation from where it stopped. Do not repeat steps that were completed.")

print(pending_note(next(x for x in CRASH if x["write"]) if any(x["write"] for x in CRASH) else CRASH[0]))

Your session crashed during your last step. The step was sent to the system but its result was lost.
Pending step: update_reservation_flights({"reservation_id":"JG7FMM","cabin":"economy","flights":[{"flight_number":"HAT028","date":"2024-05-21"},{"flight_number":"HAT277","date":"2024-05-21"}],"payment_id":"credit_card_2929732"}): a write step; database check of flights/HAT028, flights/HAT277, reservations/JG7FMM against the checkpoint taken when the step started: reservations/JG7FMM changed, so the step took effect.
Continue the conversation from where it stopped. Do not repeat steps that were completed.


### The checkpoint graph (condition d)

Built from the run up to the crash: dialogue turns, steps, records (deduplicated, latest state) and the pending step with
its state references. A tool result is a record when it carries one of the domains' id fields
(`reservation_id`, `order_id`, `user_id`, `line_id`, …); other results stay on the step node.

In [6]:
REC_KEYS = [("reservation_id", "reservations"), ("order_id", "orders"), ("line_id", "lines"), ("bill_id", "bills"),
            ("product_id", "products"), ("customer_id", "customers"), ("user_id", "users"), ("flight_number", "flights")]

def record_of(content):
    try:
        x = json.loads(content)
    except (json.JSONDecodeError, TypeError):
        return None
    if isinstance(x, dict):
        for k, coll in REC_KEYS:
            if isinstance(x.get(k), str):
                return f"{coll}/{x[k]}", x
    return None

def checkpoint_graph(x):
    ref, G = x["ref"], nx.MultiDiGraph()
    G.add_node("session", kind="session", task=ref["id"])
    last, n_step, open_calls = "session", 0, {}
    for i, ev in enumerate(ref["events"][:x["e"]]):
        if ev["role"] in ("assistant", "user") and not ev["tool_calls"] and ev["content"]:
            n = f"turn:{i}"
            G.add_node(n, kind="turn", speaker="Agent" if ev["role"] == "assistant" else "Customer", text=ev["content"])
            G.add_edge(last, n, rel="NEXT"); last = n
        elif ev["role"] == "assistant" and ev["tool_calls"]:
            for t in ev["tool_calls"]:
                n_step += 1; n = f"step:{n_step}"
                G.add_node(n, kind="step", tool=t["name"], args=t["raw"], status="completed", result=None)
                G.add_edge(last, n, rel="THEN"); open_calls[t["id"]] = n
        elif ev["role"] == "tool" and ev["requestor"] == "assistant" and ev["id"] in open_calls:
            n = open_calls.pop(ev["id"])
            rec = None if ev.get("error") else record_of(ev["content"])
            if ev.get("error"):
                G.nodes[n].update(status="failed", result=ev["content"])
            elif rec:
                rid, data = rec
                G.add_node(f"rec:{rid}", kind="record", data=data)               # latest state of the record wins
                G.add_edge(n, f"rec:{rid}", rel="WROTE" if G.nodes[n]["tool"] in WRITE else "READ")
                G.nodes[n]["result"] = f"record {rid}"
            else:
                G.nodes[n]["result"] = ev["content"]
    for t in x["calls"]:
        n_step += 1; n = f"step:{n_step}"
        G.add_node(n, kind="step", tool=t["name"], args=t["raw"], status="pending", result=None)
        G.add_edge(last, n, rel="THEN")
        for rid, data in NAMED[(x["cid"], t["id"])][0].items():
            if f"rec:{rid}" not in G:
                G.add_node(f"rec:{rid}", kind="record", data=data)
            G.add_edge(n, f"rec:{rid}", rel="STATE_REF", content=data)
    return G

def serialize(G):
    turns = [d for _, d in G.nodes(data=True) if d["kind"] == "turn"]
    steps = [(n, d) for n, d in G.nodes(data=True) if d["kind"] == "step"]
    recs = [(n, d) for n, d in G.nodes(data=True) if d["kind"] == "record"]
    s = ["CHECKPOINT GRAPH of your previous session (it crashed; the conversation with the customer is still open).",
         "\nDialogue so far:"] + [f"  {d['speaker']}: {d['text']}" for d in turns]
    s += ["\nSteps (tool calls):"]
    for n, d in steps:
        s.append(f"  {n.split(':')[1]}. [{d['status']}] {d['tool']}({d['args']})" + (f" -> {d['result']}" if d["result"] else ""))
    s += ["\nRecords (latest known state; READ / WROTE edges from the steps above):"]
    s += [f"  {n[4:]}: {json.dumps(d['data'])}" for n, d in recs]
    return "\n".join(s)

G0 = checkpoint_graph(CRASH[0])
print(f"example checkpoint graph: {G0.number_of_nodes()} nodes, {G0.number_of_edges()} edges,",
      dict(Counter(d["kind"] for _, d in G0.nodes(data=True))))
print(serialize(G0)[:1500])

example checkpoint graph: 8 nodes, 7 edges, {'session': 1, 'turn': 2, 'step': 3, 'record': 2}
CHECKPOINT GRAPH of your previous session (it crashed; the conversation with the customer is still open).

Dialogue so far:
  Agent: Hi! How can I help you today?
  Customer: Hi, I’d like to cancel reservation EHGLP3. I’m Emma Kim.

Steps (tool calls):
  1. [completed] get_reservation_details({"reservation_id":"EHGLP3"}) -> record reservations/EHGLP3
  2. [completed] get_flight_status({"flight_number":"HAT156","date":"2024-05-17"}) -> available
  3. [pending] get_flight_status({"flight_number":"HAT021","date":"2024-05-17"})

Records (latest known state; READ / WROTE edges from the steps above):
  reservations/EHGLP3: {"reservation_id": "EHGLP3", "user_id": "emma_kim_9957", "origin": "PHX", "destination": "JFK", "flight_type": "one_way", "cabin": "basic_economy", "flights": [{"flight_number": "HAT156", "origin": "PHX", "destination": "SEA", "date": "2024-05-17", "price": 50}, {"flight_number": 

## Resuming: four conditions per crash point, one LLM round per step

In [7]:
ARMS = ("restart", "history", "history_note", "graph")
RESTART_GREETING = "I'm sorry, we were disconnected and I lost the record of our conversation. Hi! How can I help you today?"

def resumed(x, arm):
    ref, e = x["ref"], x["e"]
    c = new_conv(ref["key"], cid=f"{x['cid']}|{arm}")
    c.update(events=[dict(ev) for ev in ref["events"][:e]], log=list(x["crash_log"]), steps=0, resume_at=e,
             turn="agent", crash=x, arm=arm)
    if arm == "restart":
        c["events"].append({"role": "assistant", "content": RESTART_GREETING, "tool_calls": []})
        c["agent_prefix"], c["turn"] = [], "user"
    elif arm == "history":
        c["agent_prefix"] = ref["events"][:e]
    elif arm == "history_note":
        c["agent_prefix"] = ref["events"][:e] + [{"role": "note", "content": pending_note(x)}]
    else:
        c["agent_prefix"] = [{"role": "note", "content": serialize(checkpoint_graph(x)) + "\n\n" + pending_note(x)}]
    return c

RUNS = [resumed(x, a) for x in CRASH for a in ARMS]
t0 = time.time()
first_prompt = {}
def note_first(n, live):
    if n % 10 == 0:
        print(f"round {n}: {len(live)} live")
rounds = simulate(RUNS, "nb27_resume", on_step=note_first)
print(f"{len(RUNS)} resumed runs ({len(CRASH)} crash points x {len(ARMS)}) in {rounds} rounds, {time.time() - t0:.0f} s")

[nb27_resume_step1] 1,092 new requests -> test (0 cached)


[nb27_resume_step2] 1,090 new requests -> test (0 cached)


[nb27_resume_step3] 1,079 new requests -> test (0 cached)


[nb27_resume_step4] 981 new requests -> test (0 cached)


[nb27_resume_step5] 944 new requests -> test (0 cached)


[nb27_resume_step6] 907 new requests -> test (0 cached)


... (48 outputs left out)

[nb27_resume_step55] 1 new requests -> test (0 cached)


[nb27_resume_step56] 1 new requests -> test (0 cached)


[nb27_resume_step57] 1 new requests -> test (0 cached)


1092 resumed runs (273 crash points x 4) in 57 rounds, 25974 s


## Scores: τ²-bench's state evaluation of the resumed runs, repeated steps, tokens

In [8]:
items = []
for c in RUNS:
    x = c["crash"]
    pre = x["ref"]["events"][:x["e"] + 1 + len(x["calls"])]          # recorded part, including the crashed step
    msgs = []
    for ev in pre + c["events"][x["e"]:]:
        if ev["role"] in ("assistant", "user"):
            msgs.append({"role": ev["role"], "content": ev["content"],
                         "tool_calls": [{"id": t["id"], "name": t["name"], "arguments": t["arguments"]}
                                        for t in ev["tool_calls"] if t["arguments"] is not None]})
        elif ev["role"] == "tool":
            msgs.append({"role": "tool", "id": ev["id"], "content": ev["content"], "requestor": ev["requestor"]})
    items.append({"domain": c["domain"], "task": c["task"], "log": c["log"], "messages": msgs})
EV = srv.ask({"op": "evaluate", "items": items})["results"]

canon = lambda t: (t["name"], json.dumps(t["arguments"], sort_keys=True))
for c, ev in zip(RUNS, EV):
    x = c["crash"]
    before = {canon(t) for e2 in x["ref"]["events"][:x["e"] + 1] if e2["role"] == "assistant"
              for t in e2["tool_calls"] if t["arguments"] is not None}
    after = [t for e2 in c["events"][x["e"]:] if e2["role"] == "assistant" for t in e2["tool_calls"] if t["arguments"] is not None]
    rep = [t for t in after if canon(t) in before]
    c["m"] = {"success": float(ev["reward"] if c["term"] == "user_stop" else 0.0),
              "repeated": len(rep), "repeated_writes": sum(t["name"] in WRITE for t in rep),
              "post_calls": len(after),
              "tokens": sum(c["tokens"].values()), "agent_tokens": c["tokens"]["agent_in"] + c["tokens"]["agent_out"],
              "resume_prompt": c.get("first_agent_in", np.nan), "term": c["term"]}

BY = {a: [c for c in RUNS if c["arm"] == a] for a in ARMS}   # same crash-point order in every arm
LAB = {"restart": "(a) restart", "history": "(b) message history", "history_note": "(c) history + checkpoint note",
       "graph": "(d) checkpoint graph"}
SUMMARY = {}
def table(label, sel):
    idx = [i for i, x in enumerate(CRASH) if sel(x)]
    if not idx:
        return
    print(f"\n{label}  (n = {len(idx)} crash points)")
    print(f"{'condition':<32}{'success':>9}{'95% CI':>16}{'repeated steps':>16}{'repeated writes':>17}{'tokens after':>14}{'resume prompt':>15}")
    for a in ARMS:
        ms = [BY[a][i]["m"] for i in idx]
        k = sum(m["success"] for m in ms); lo, hi = wilson(k, len(ms))
        SUMMARY[(label, a)] = {key: np.nanmean([m[key] for m in ms]) for key in ("success", "repeated", "repeated_writes", "tokens", "resume_prompt")}
        s = SUMMARY[(label, a)]
        print(f"{LAB[a]:<32}{s['success']:>9.1%}{f'[{lo:.0%}, {hi:.0%}]':>16}{s['repeated']:>16.2f}{s['repeated_writes']:>17.2f}"
              f"{s['tokens']:>14,.0f}{s['resume_prompt']:>15,.0f}")

table("all crash points", lambda x: True)
table("crash at a write step", lambda x: x["write"])
table("crash at a read step", lambda x: not x["write"])
for d in DOM:
    table(d, lambda x, d=d: x["ref"]["domain"] == d)
print("\nEnd reasons:", {a: dict(Counter(c["m"]["term"] for c in BY[a])) for a in ARMS})


all crash points  (n = 273 crash points)
condition                         success          95% CI  repeated steps  repeated writes  tokens after  resume prompt
(a) restart                         79.1%      [74%, 84%]            3.56             0.02        67,665          4,248
(b) message history                 83.2%      [78%, 87%]            1.31             0.11        54,348          5,331
(c) history + checkpoint note       85.3%      [81%, 89%]            1.36             0.00        52,063          5,423
(d) checkpoint graph                81.3%      [76%, 85%]            1.45             0.00        58,469          5,680

crash at a write step  (n = 32 crash points)
condition                         success          95% CI  repeated steps  repeated writes  tokens after  resume prompt
(a) restart                         87.5%      [72%, 95%]            5.31             0.12        47,451          4,115
(b) message history                 90.6%      [76%, 97%]            1.5

### Paired differences (95% paired bootstrap over crash points)

In [9]:
PAIRS = [("graph", "history_note"), ("graph", "history"), ("graph", "restart"), ("history_note", "history"), ("history", "restart")]
DIFF = {}
print(f"{'comparison':<44}{'success':>24}{'repeated steps':>26}{'tokens after resume':>30}")
for a, b in PAIRS:
    row = []
    for key, fmt in (("success", "{:+.1%} [{:+.1%}, {:+.1%}]"), ("repeated", "{:+.2f} [{:+.2f}, {:+.2f}]"),
                     ("tokens", "{:+,.0f} [{:+,.0f}, {:+,.0f}]")):
        t = paired_boot([c["m"][key] for c in BY[b]], [c["m"][key] for c in BY[a]]) if CRASH else (np.nan,) * 3
        DIFF[(a, b, key)] = t
        row.append(fmt.format(*t))
    print(f"{LAB[a] + ' - ' + LAB[b][:3]:<44}{row[0]:>24}{row[1]:>26}{row[2]:>30}"
          + ("   <- primary" if (a, b) == ("graph", "history_note") else ""))

comparison                                                   success            repeated steps           tokens after resume
(d) checkpoint graph - (c)                      -4.0% [-8.8%, +0.7%]      +0.10 [-0.04, +0.23]        +6,406 [-853, +15,500]   <- primary
(d) checkpoint graph - (b)                      -1.8% [-7.3%, +3.7%]      +0.14 [-0.04, +0.32]      +4,121 [-5,249, +13,932]
(d) checkpoint graph - (a)                      +2.2% [-2.9%, +7.7%]      -2.11 [-2.45, -1.80]        -9,197 [-17,187, -239]
(c) history + checkpoint note - (b)             +2.2% [-2.9%, +7.0%]      +0.05 [-0.08, +0.18]      -2,286 [-11,817, +5,368]
(b) message history - (a)                       +4.0% [-1.5%, +9.5%]      -2.25 [-2.58, -1.96]     -13,317 [-17,938, -7,751]


### One crash point, four resumptions

In [10]:
x = next((x for x in CRASH if x["write"]), CRASH[0])
i = CRASH.index(x)
print(f"{x['cid']}: crash at step {x['k']} of {x['n_steps']}: " + ", ".join(f"{t['name']}({t['raw']})" for t in x["calls"]))
for a in ARMS:
    c = BY[a][i]
    calls = [f"{t['name']}" for e2 in c["events"][x["e"]:] if e2["role"] == "assistant" for t in e2["tool_calls"]]
    print(f"\n{LAB[a]}: success {c['m']['success']:.0f}, end {c['term']}, repeated {c['m']['repeated']}, "
          f"tokens {c['m']['tokens']:,}\n   steps after the resume: {calls[:12]}")
    first_agent = next((e2 for e2 in c["events"][x["e"]:] if e2["role"] == "assistant"), None)
    if first_agent:
        print(f"   first agent turn: {(first_agent['content'] or '')[:200]!r} "
              f"{[t['name'] for t in first_agent['tool_calls']]}")

airline|18|e28: crash at step 4 of 9: update_reservation_flights({"reservation_id":"JG7FMM","cabin":"economy","flights":[{"flight_number":"HAT028","date":"2024-05-21"},{"flight_number":"HAT277","date":"2024-05-21"}],"payment_id":"credit_card_2929732"})

(a) restart: success 0, end user_stop, repeated 17, tokens 54,636
   steps after the resume: ['get_user_details', 'get_reservation_details', 'get_reservation_details', 'get_reservation_details', 'get_reservation_details', 'get_reservation_details', 'get_reservation_details', 'get_flight_status', 'get_flight_status', 'get_flight_status', 'get_flight_status', 'get_flight_status']
   first agent turn: "I'm sorry, we were disconnected and I lost the record of our conversation. Hi! How can I help you today?" []

(b) message history: success 1, end user_stop, repeated 1, tokens 63,933
   steps after the resume: ['update_reservation_flights', 'update_reservation_flights', 'update_reservation_flights', 'update_reservation_flights', 'update_rese

## What this run cost, and a forecast of the full run

The forecast scales the measured tokens per resumed run (per domain and condition) to the full run: 2 crash points per
successful reference run of all 278 tasks, the success rate of the reference runs taken from Notebook 4.4's recordings
(in a test run: from the test recordings).

In [11]:
STAGES = sorted(f[:-6] for f in os.listdir(llm.cache) if f.startswith("nb27_") and f.endswith(".jsonl"))
LEDGER = llm.ledger(STAGES)
COST = sum(a["usd"] for a in LEDGER.values() if a["calls"])
print(f"resume rounds: {len(STAGES)}; calls {sum(LEDGER[s]['calls'] for s in STAGES):,}; input tokens "
      f"{sum(LEDGER[s]['in'] for s in STAGES):,}; output tokens {sum(LEDGER[s]['out'] for s in STAGES):,}; USD {COST:.2f}; "
      f"via Batch {np.mean([LEDGER[s]['batch_share'] for s in STAGES]):.0%}")

FULL_N = Counter(t["domain"] for t in META["tasks"])
price = lambda i, o, tier: (i * {"batch": 0.05, "standard": 0.10}[tier] + o * {"batch": 0.25, "standard": 0.50}[tier]) / 1e6
fc, n_calls, n_cp = {"batch": 0.0, "standard": 0.0}, 0, 0
for d in DOM:
    runs = [c for c in RUNS if c["domain"] == d]
    if not runs:
        continue
    succ = np.mean([c["reward"] for c in REC if c["domain"] == d])
    cps = N_CRASH * FULL_N[d] * succ
    n_cp += cps
    for a in ARMS:
        rs = [c for c in runs if c["arm"] == a]
        i = np.mean([c["tokens"]["agent_in"] + c["tokens"]["user_in"] for c in rs]) * cps
        o = np.mean([c["tokens"]["agent_out"] + c["tokens"]["user_out"] for c in rs]) * cps
        n_calls += np.mean([c["llm_calls"] for c in rs]) * cps
        for t in fc:
            fc[t] += price(i, o, t)
print(f"Forecast, full run: ~{n_cp:,.0f} crash points x {len(ARMS)} conditions, ~{n_calls:,.0f} calls, "
      f"${fc['batch']:.2f} via Batch (${fc['standard']:.2f} standard).")
print("cost_forecast.py, '5.3': 43,368 calls, $25.48 standard (≈ $13 via Batch), for 278 x 2 crash points incl. the run to the crash.")

resume rounds: 57; calls 12,543; input tokens 62,911,403; output tokens 573,324; USD 6.58; via Batch 0%
Forecast, full run: ~276 crash points x 4 conditions, ~12,664 calls, $3.32 via Batch ($6.65 standard).
cost_forecast.py, '5.3': 43,368 calls, $25.48 standard (≈ $13 via Batch), for 278 x 2 crash points incl. the run to the crash.


## Limitations

- **Successful reference runs only.** As in Crab, crashes are injected only into runs the model completed without a
  fault, so resumption success is measured on tasks the model can solve; the share of tasks excluded is printed above.
- **One crash model.** The crash hits after the environment executed a tool step and before the agent stored its result
  (the case where state and memory disagree). Crashes during a model call, lost user messages, or a lost database are not
  simulated.
- **The user simulator is an LLM.** After a restart it must restate its request; it can drift or end the conversation
  early, which counts against restart.
- **Checkpoint content.** The graph's record extraction uses the id fields of the three domains; the pending-step check
  reads the named records from the database (the stored state references), which a deployed system would do through its
  read tools. Condition (c) receives the same check as text, so (d) − (c) isolates the graph form (records once, at their
  latest state, instead of every tool output), not the extra information.
- **Repeated steps** count identical calls (same tool and arguments) only; a re-search with different arguments is not
  counted.
- **One model** (`gpt-6-luna`, no reasoning), one seed per crash point.

**References.** Wu et al., *Crab: A Semantics-Aware Checkpoint/Restore Runtime for Agent Sandboxes*, arXiv 2604.28138
(2026) · Zhuang et al., *AgentRewind: Recoverable Execution for Long-Horizon LLM Agents*, arXiv 2608.14380 (2026) ·
Barres et al., *τ²-Bench*, arXiv 2506.07982 (2025) · LangGraph checkpointing (Neo4jSaver).

In [12]:
print("=" * 118)
print(f"RESULTS SUMMARY — Notebook 4.3: execution state and recovery on τ²-bench (commit {TAU2_COMMIT[:7]}), {MODE} run, gpt-6-luna")
print("=" * 118)
print(f"Reference runs: {len(REF)} successful of {len(REC)} recorded; crash points: {len(CRASH)} "
      f"({sum(x['write'] for x in CRASH)} at a write step)")
print(f"\n{'condition':<32}{'success':>9}{'repeated steps':>16}{'repeated writes':>17}{'tokens after':>14}{'resume prompt':>15}")
for a in ARMS:
    s = SUMMARY[("all crash points", a)]
    print(f"{LAB[a]:<32}{s['success']:>9.1%}{s['repeated']:>16.2f}{s['repeated_writes']:>17.2f}{s['tokens']:>14,.0f}{s['resume_prompt']:>15,.0f}")
d, lo, hi = DIFF[("graph", "history_note", "success")]
verdict = "checkpoint graph better" if lo > 0 else "checkpoint graph worse" if hi < 0 else "no resolvable difference"
print(f"\nPRIMARY  success(checkpoint graph) - success(history + same checkpoint facts) = {d:+.1%} [{lo:+.1%}, {hi:+.1%}]  ->  {verdict}")
for a, b in PAIRS[1:]:
    d, lo, hi = DIFF[(a, b, "success")]
    print(f"         success {LAB[a]} - {LAB[b]} = {d:+.1%} [{lo:+.1%}, {hi:+.1%}]")
d, lo, hi = DIFF[("graph", "history_note", "tokens")]
print(f"         tokens after the resume, (d) - (c) = {d:+,.0f} [{lo:+,.0f}, {hi:+,.0f}]")
print(f"Cost of this run: ${COST:.2f}")
print("\nLecture message (Task 4.3): what a crashed agent needs is the state of the step in flight, not the whole"
      "\nconversation; whether a checkpoint graph resumes as reliably as the full history is the PRIMARY line.")

RESULTS SUMMARY — Notebook 4.3: execution state and recovery on τ²-bench (commit 96a3ef9), batch run, gpt-6-luna
Reference runs: 138 successful of 278 recorded; crash points: 273 (32 at a write step)

condition                         success  repeated steps  repeated writes  tokens after  resume prompt
(a) restart                         79.1%            3.56             0.02        67,665          4,248
(b) message history                 83.2%            1.31             0.11        54,348          5,331
(c) history + checkpoint note       85.3%            1.36             0.00        52,063          5,423
(d) checkpoint graph                81.3%            1.45             0.00        58,469          5,680

PRIMARY  success(checkpoint graph) - success(history + same checkpoint facts) = -4.0% [-8.8%, +0.7%]  ->  no resolvable difference
         success (d) checkpoint graph - (b) message history = -1.8% [-7.3%, +3.7%]
         success (d) checkpoint graph - (a) restart = +2.2% [-2.